In [ ]:
import os, sys, re
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, r"c:\repos\DroneDetectionRF")
from NoisyUAV.funciones.dsp_rf.detector_entropia import detectar_bursts

In [ ]:

# Rutas
CSV_PATH     = r"c:\repos\DroneDetectionRF\NoisyUAV\modelo_alumn_v3_dual_pro\dataset_v5_pointers.csv"
RAW_DIR      = r"C:\TFM_data\NoisyUAV\drone_RF_data"
OUT_CSV      = r"c:\repos\DroneDetectionRF\NoisyUAV\modelo_alumn_v3_dual_pro\dataset_v5_burst_features.csv"
TARGET_NOISE = 4
# Parámetros CFAR canónicos (idénticos al builder V2)
FS       = 14e6
NPERSEG  = 2048
Z_THRESH = 4.0
print("✅ Imports OK")

In [ ]:
df = pd.read_csv(CSV_PATH)
ficheros_unicos = df['filename'].unique()
print(f"CSV cargado: {len(df)} instancias | {len(ficheros_unicos)} ficheros únicos")

records = []
n_sin_bursts = 0
n_error = 0

for fname in tqdm(ficheros_unicos, desc="Extrayendo features por burst"):
    fpath = os.path.join(RAW_DIR, fname)
    if not os.path.exists(fpath):
        n_error += 1
        continue

    try:
        d  = torch.load(fpath, map_location='cpu', weights_only=False)
        iq = d['x_iq'].float()
    except Exception:
        n_error += 1
        continue
    # Target y SNR siempre desde el nombre del fichero (igual que el builder)
    m_fname = re.match(r"IQdata_sample\d+_target(\d+)_snr(-?\d+)\.pt", fname)
    if m_fname is None:
        n_error += 1
        continue
    target = int(m_fname.group(1))
    snr    = int(m_fname.group(2))
    label  = 0 if target == TARGET_NOISE else 1

    try:
        t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
            iq, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
            min_burst_ms=0.5, merge_gap_ms=0.75, min_z_abs=3.5,
            bg_mult=4, max_bins_frac=1.0, smooth_ms=0.3, adaptive_window_ms=10
        )
    except Exception:
        n_error += 1
        continue

    if len(bursts) == 0:
        n_sin_bursts += 1
        continue

    # Para cada burst, extraer features físicas
    for b in bursts:
        t0, t1 = b['t0'], b['t1']
        duration_ms = t1 - t0

        # Índices de t_ms dentro del burst
        mask = (t_ms >= t0) & (t_ms <= t1)
        n_bins_in_burst = n_active[mask] if np.any(mask) else np.array([0])

        n_bins_peak  = float(np.max(n_bins_in_burst))
        n_bins_mean  = float(np.mean(n_bins_in_burst))
        frac_espectro = n_bins_peak / NPERSEG  # 0..1

        records.append({
            'filename':         fname,
            'target':           target,
            'label':            label,           # 0=ruido, 1=dron
            'snr':              snr,
            'split':            df.loc[df['filename'] == fname, 'split'].iloc[0],
            # Features CFAR globales
            'global_nf':        float(np.median(nf_v)),
            'global_H_mean':    float(np.mean(H_smooth)),
            # Features por burst
            't0_ms':            t0,
            't1_ms':            t1,
            'duration_ms':      duration_ms,
            'z_peak':           abs(b['z_peak']),
            'n_bins_peak':      n_bins_peak,
            'n_bins_mean':      n_bins_mean,
            'frac_espectro':    frac_espectro,
        })

df_bursts = pd.DataFrame(records)
df_bursts.to_csv(OUT_CSV, index=False)
print(f"\n✅ Features extraídas: {len(df_bursts)} bursts de {len(ficheros_unicos) - n_error} ficheros")
print(f"   Sin bursts: {n_sin_bursts} | Errores: {n_error}")
print(f"   Guardado en: {OUT_CSV}")


In [ ]:
df_bursts = pd.read_csv(OUT_CSV)  # Recargar si ya existe

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Distribución de Features por Burst — Exploración Pre-HDBSCAN", fontsize=14)

colores = {0: 'steelblue', 1: 'tomato'}
etiquetas = {0: 'Ruido (label=0)', 1: 'Dron (label=1)'}

for label_val, color in colores.items():
    sub = df_bursts[df_bursts['label'] == label_val]
    kw = dict(alpha=0.4, bins=60, color=color, label=etiquetas[label_val])

    axes[0, 0].hist(sub['n_bins_peak'],   **kw); axes[0, 0].set_title("n_bins_peak"); axes[0, 0].set_xlabel("Bins activos (pico)")
    axes[0, 1].hist(sub['duration_ms'],   **kw); axes[0, 1].set_title("Duración burst (ms)"); axes[0, 1].set_xlabel("ms")
    axes[0, 2].hist(sub['z_peak'],        **kw); axes[0, 2].set_title("z_peak (|z|)"); axes[0, 2].set_xlabel("|z|")
    axes[1, 0].hist(sub['frac_espectro'], **kw); axes[1, 0].set_title("Fracción espectro"); axes[1, 0].set_xlabel("[0, 1]")
    axes[1, 1].hist(sub['n_bins_mean'],   **kw); axes[1, 1].set_title("n_bins_mean"); axes[1, 1].set_xlabel("Bins medios")
    axes[1, 2].hist(np.log1p(sub['n_bins_peak']), **kw); axes[1, 2].set_title("log(1 + n_bins_peak)"); axes[1, 2].set_xlabel("log scale")

for ax in axes.flat:
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print("\nEstadísticas por label:")
print(df_bursts.groupby('label')[['n_bins_peak','duration_ms','z_peak','frac_espectro']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel izquierdo: coloreado por label (dron/ruido)
for label_val, color, etiq in [(0, 'steelblue', 'Ruido'), (1, 'tomato', 'Dron')]:
    sub = df_bursts[df_bursts['label'] == label_val]
    axes[0].scatter(sub['duration_ms'], sub['n_bins_peak'],
                    c=color, alpha=0.15, s=5, label=etiq)
axes[0].set_xlabel("Duración burst (ms)"); axes[0].set_ylabel("n_bins_peak")
axes[0].set_title("Espacio de features: por label")
axes[0].axhline(2048*0.5, color='k', linestyle='--', alpha=0.4, label='50% espectro')
axes[0].axhline(400, color='orange', linestyle='--', alpha=0.4, label='400 bins (posible umbral FHSS)')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 15); axes[0].set_ylim(0, 2200)

# Panel derecho: coloreado por target (0-6)
cmap = plt.get_cmap('tab10')
for target_val in sorted(df_bursts['target'].unique()):
    sub = df_bursts[df_bursts['target'] == target_val]
    axes[1].scatter(sub['duration_ms'], sub['n_bins_peak'],
                    c=[cmap(target_val/7)], alpha=0.2, s=5, label=f"Target {target_val}")
axes[1].set_xlabel("Duración burst (ms)"); axes[1].set_ylabel("n_bins_peak")
axes[1].set_title("Espacio de features: por target (modelo de emisor)")
axes[1].legend(fontsize=8, markerscale=3); axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 15); axes[1].set_ylim(0, 2200)

plt.tight_layout(); plt.show()


In [ ]:
from sklearn.preprocessing import StandardScaler
try:
    import hdbscan
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'hdbscan'])
    import hdbscan

# Features para clustering (las más discriminativas físicamente)
FEATURE_COLS = ['n_bins_peak', 'duration_ms', 'z_peak', 'frac_espectro', 'n_bins_mean']

X = df_bursts[FEATURE_COLS].values
X_scaled = StandardScaler().fit_transform(X)

print(f"Ejecutando HDBSCAN sobre {len(X)} bursts con features {FEATURE_COLS}...")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=100,   # clústeres mínimos de 100 bursts
    min_samples=20,
    metric='euclidean',
    cluster_selection_method='eom'
)
cluster_labels = clusterer.fit_predict(X_scaled)

df_bursts['cluster'] = cluster_labels
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_outliers = np.sum(cluster_labels == -1)

print(f"\n✅ HDBSCAN completado:")
print(f"   Clústeres encontrados: {n_clusters}")
print(f"   Outliers (cluster=-1): {n_outliers} ({n_outliers/len(X)*100:.1f}%)")

print("\nComposición de cada clúster:")
for c in sorted(df_bursts['cluster'].unique()):
    sub = df_bursts[df_bursts['cluster'] == c]
    frac_dron = sub['label'].mean()
    n_bins_med = sub['n_bins_peak'].median()
    dur_med = sub['duration_ms'].median()
    n = len(sub)
    nombre_cluster = "OUTLIER" if c == -1 else f"Cluster {c}"
    print(f"  [{nombre_cluster:12s}] N={n:5d} | n_bins_median={n_bins_med:6.0f} | dur_median={dur_med:.2f}ms | %dron={frac_dron*100:.1f}%")


In [ ]:
try:
    import umap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn'])
    import umap

print("Calculando proyección UMAP (puede tardar ~1 min)...")
reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, n_components=2, random_state=42)
embedding = reducer.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("UMAP del espacio de bursts RF — Separación de fuentes", fontsize=14)

# Panel 1: coloreado por clúster HDBSCAN
n_c = max(cluster_labels) + 2
cmap_c = plt.get_cmap('tab10')
for c in sorted(set(cluster_labels)):
    mask = cluster_labels == c
    color = 'gray' if c == -1 else cmap_c(c / max(1, n_clusters))
    label = "Outlier" if c == -1 else f"Cluster {c}"
    axes[0].scatter(embedding[mask, 0], embedding[mask, 1],
                    c=[color], alpha=0.3, s=4, label=label)
axes[0].set_title("Por Cluster HDBSCAN"); axes[0].legend(fontsize=7, markerscale=3)

# Panel 2: coloreado por label (dron/ruido)
for lv, color, etiq in [(0, 'steelblue', 'Ruido'), (1, 'tomato', 'Dron')]:
    mask = df_bursts['label'].values == lv
    axes[1].scatter(embedding[mask, 0], embedding[mask, 1],
                    c=color, alpha=0.2, s=4, label=etiq)
axes[1].set_title("Por Label (GT)"); axes[1].legend(fontsize=8, markerscale=3)

# Panel 3: coloreado por n_bins_peak (la feature clave)
sc = axes[2].scatter(embedding[:, 0], embedding[:, 1],
                     c=np.log1p(df_bursts['n_bins_peak'].values),
                     cmap='plasma', alpha=0.3, s=4)
plt.colorbar(sc, ax=axes[2], label='log(1 + n_bins_peak)')
axes[2].set_title("Por log(n_bins_peak)")

for ax in axes:
    ax.grid(True, alpha=0.2); ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")

plt.tight_layout(); plt.show()


In [ ]:
print("=" * 70)
print("ANÁLISIS DE LABEL NOISE POR CLÚSTER")
print("=" * 70)
print("\nHipótesis: los clústeres con n_bins_peak >> 400 son WiFi/BT (label noise)")
print("           los clústeres con n_bins_peak ~186 son FHSS-dron (label correcto)")
print()

for c in sorted(df_bursts['cluster'].unique()):
    if c == -1: continue
    sub = df_bursts[df_bursts['cluster'] == c]

    # ¿Cuántos bursts de este clúster están en ficheros de dron pero podrían ser WiFi/BT?
    sub_en_dron  = sub[sub['label'] == 1]  # en ficheros de dron
    sub_en_ruido = sub[sub['label'] == 0]  # en ficheros de ruido

    n_bins_p50 = sub['n_bins_peak'].quantile(0.50)
    dur_p50    = sub['duration_ms'].quantile(0.50)

    # Clasificación tentativa del clúster
    if n_bins_p50 > 1000:
        tipo = "PROBABLE WIFI (broadband)"
        label_noise = len(sub_en_dron)
    elif n_bins_p50 < 100:
        tipo = "PROBABLE BT (narrowband)"
        label_noise = len(sub_en_dron)
    else:
        tipo = "PROBABLE FHSS-DRON o RUIDO"
        label_noise = 0

    print(f"Cluster {c:2d} | N={len(sub):5d} | n_bins_p50={n_bins_p50:6.0f} | dur_p50={dur_p50:.2f}ms")
    print(f"          | Tipo: {tipo}")
    print(f"          | En ficheros dron: {len(sub_en_dron)} | En ficheros ruido: {len(sub_en_ruido)}")
    if label_noise > 0:
        print(f"          | ⚠️  LABEL NOISE ESTIMADO: {label_noise} bursts de dron mal etiquetados")
    print()

total_label_noise = len(df_bursts[
    (df_bursts['label'] == 1) &
    (df_bursts['n_bins_peak'] > 1000)  # proxy WiFi
])
total_dron = len(df_bursts[df_bursts['label'] == 1])
print(f"ESTIMACIÓN GLOBAL: {total_label_noise}/{total_dron} instancias de dron son potencial label noise")
print(f"                   ({total_label_noise/total_dron*100:.1f}% del training set de drones)")
